In [84]:
import math
import pandas as pd


DB_PATH = "runs_index.csv"

DB = pd.read_csv(DB_PATH, index_col=0)


PROBLEMS = ['vanderpol', 'pollu', 'rober', 'orego', 'hires', 'davis-skodje']
PRETRAINING_OPTIONS = ['derivmatch', 'none']
TRAINING_OPTIONS = ['shooting', 'collocation', 'none']
SEED_OPTIONS = list(range(10, 21))  # seeds from 10 to 20 inclusive

DB.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1719 entries, run_03lcWbxB to run_zzeNEZ2z
Data columns (total 46 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   E_VF_species_test           1141 non-null   object 
 1   E_VF_species_train          1141 non-null   object 
 2   E_VF_test                   1090 non-null   float64
 3   E_VF_train                  1135 non-null   float64
 4   E_spec_species_test         1141 non-null   object 
 5   E_spec_species_train        1141 non-null   object 
 6   E_spec_test                 1090 non-null   float64
 7   E_spec_train                1135 non-null   float64
 8   E_trajectory_species_test   1141 non-null   object 
 9   E_trajectory_species_train  1141 non-null   object 
 10  E_trajectory_test           1090 non-null   float64
 11  E_trajectory_train          1135 non-null   float64
 12  collocation_time            1719 non-null   float64
 13  computer           

### Preprocess

In [85]:
# filter out all with E_vf_specites_train not being present, means of removing old runs. 
DB = DB[DB['E_VF_test'].notnull()]

# Remove rows where seed is not between 10 and 20 (inclusive)
DB = DB[(DB['seed'] >= 20) & (DB['seed'] <= 29)]

for problem in PROBLEMS: 
    for pretraining in PRETRAINING_OPTIONS:
        for training in TRAINING_OPTIONS:
            for seed in SEED_OPTIONS:
                # Filter the DataFrame for the current combination of problem, pretraining, training, and seed
                filtered_rows = DB[(DB['problem'] == problem) & 
                                   (DB['pretraining'] == pretraining) & 
                                   (DB['training'] == training) & 
                                   (DB['seed'] == seed)]
                
                # If there are more than two rows with the same combination, print the details
                if len(filtered_rows) > 2:

                    print(f"Problem: {problem}, Pretraining: {pretraining}, Training: {training}, Seed: {seed}, Count: {len(filtered_rows)}")

                # Exit all loops after finding the first duplicate for this combination

Problem: vanderpol, Pretraining: derivmatch, Training: shooting, Seed: 20, Count: 3
Problem: vanderpol, Pretraining: derivmatch, Training: collocation, Seed: 20, Count: 3
Problem: pollu, Pretraining: derivmatch, Training: collocation, Seed: 20, Count: 3
Problem: rober, Pretraining: derivmatch, Training: shooting, Seed: 20, Count: 3
Problem: orego, Pretraining: derivmatch, Training: shooting, Seed: 20, Count: 3
Problem: orego, Pretraining: derivmatch, Training: collocation, Seed: 20, Count: 3
Problem: hires, Pretraining: derivmatch, Training: shooting, Seed: 20, Count: 3
Problem: hires, Pretraining: derivmatch, Training: collocation, Seed: 20, Count: 3
Problem: davis-skodje, Pretraining: derivmatch, Training: shooting, Seed: 20, Count: 3
Problem: davis-skodje, Pretraining: derivmatch, Training: collocation, Seed: 20, Count: 3


In [86]:
problem = "vanderpol"
model = "GELU-scaled"
for problem in PROBLEMS:
    for model in DB["model"].unique():
        print(f"Problem: {problem}, Model: {model}, Unique n_params: {DB[(DB['problem'] == problem) & (DB['model'] == model)]['n_params'].unique()}")
    print()

Problem: vanderpol, Model: mlp, Unique n_params: [282]
Problem: vanderpol, Model: stiff, Unique n_params: [268]
Problem: vanderpol, Model: GELU-scaled, Unique n_params: [317]

Problem: pollu, Model: mlp, Unique n_params: [4940]
Problem: pollu, Model: stiff, Unique n_params: [4744]
Problem: pollu, Model: GELU-scaled, Unique n_params: [4940]

Problem: rober, Model: mlp, Unique n_params: [348]
Problem: rober, Model: stiff, Unique n_params: [318]
Problem: rober, Model: GELU-scaled, Unique n_params: [332]

Problem: orego, Model: mlp, Unique n_params: [516]
Problem: orego, Model: stiff, Unique n_params: [486]
Problem: orego, Model: GELU-scaled, Unique n_params: [516]

Problem: hires, Model: mlp, Unique n_params: [827]
Problem: hires, Model: stiff, Unique n_params: [816]
Problem: hires, Model: GELU-scaled, Unique n_params: [855]

Problem: davis-skodje, Model: mlp, Unique n_params: [282]
Problem: davis-skodje, Model: stiff, Unique n_params: [268]
Problem: davis-skodje, Model: GELU-scaled, Uniq

In [87]:
# Filter out so that when there are two sets of parameters for the same problem+model, we only keep the one with the bigger number of parameters.
max_n_params = DB.groupby(["problem", "model"])["n_params"].transform("max")
DB = DB[DB["n_params"] == max_n_params]

make a function that takes in 'db' and 'problem'

db has the columns E_trajectory_test E_spec_test E_VF_test  seed, n_params training model 
model has three different values, training has two different values.

I want to create a table with the following rows: 
trajectory spec VF (all these are scalars), and also seed (this will be a list of seed) and n_params (list of n_params). The columns will first be the first three different models with the one training, and then again the same three models with the other training. So the columns will be: model1_training1, model2_training1, model3_training1, model1_training2, model2_training2, model3_training2.

Per model/training combination, there are different seeds. When calculating the different values, first filter out the 'k' rows with the worst E_trajectory_test, then calculate the mean of the remaining rows for E_trajectory_test, E_spec_test, and E_VF_test. For seed and n_params, just return the list of seeds and n_params for the remaining rows.

the function should return a pandas DataFrame with the specified rows and columns and hopefully also print those values nicely. 

In [97]:
def mainablation_results(db, problem, k=4, model_order=None, training_order=None):
    """Build a model x training summary table for a given problem.

    Per (model, training) combination, drops the `k` rows with the worst
    (highest) E_trajectory_test and the `k` rows with the best (lowest)
    E_trajectory_test, then averages E_trajectory_test, E_spec_test,
    E_VF_test over the remaining rows; seed and n_params are kept as lists
    of the remaining rows' values.
    """
    sub = db[db["problem"] == problem]

    if model_order is None:
        model_order = sorted(sub["model"].dropna().unique())
    if training_order is None:
        # "none" means no post-training step (pretraining-only run); exclude it
        # so the default is the two real training methods (collocation, shooting).
        training_order = sorted(t for t in sub["training"].dropna().unique() if t != "none")

    columns = [f"{model}_{training}" for training in training_order for model in model_order]
    row_names = ["trajectory", "spec", "VF", "seed", "n_params"]
    table = pd.DataFrame(index=row_names, columns=columns, dtype=object)

    for training in training_order:
        for model in model_order:
            col = f"{model}_{training}"
            grp = sub[(sub["model"] == model) & (sub["training"] == training)]
            grp = grp.sort_values("E_trajectory_test", ascending=True)  # best first
            kept = grp.iloc[k:len(grp) - k] if k > 0 else grp

            table.loc["trajectory", col] = f"{kept['E_trajectory_test'].mean():.3e}"
            table.loc["spec", col] = f"{kept['E_spec_test'].mean():.3e}"
            table.loc["VF", col] = f"{kept['E_VF_test'].mean():.3e}"
            table.loc["seed", col] = list(kept["seed"])
            table.loc["n_params", col] = list(kept["n_params"])

    # print(f"Main ablation results for problem={problem} (dropped {k} best and {k} worst E_trajectory_test run(s) per column)")
    # print(table.to_string())

    return table

for p in PROBLEMS:
    print(f"Main ablation results for problem={p} (dropped {2} best and {2} worst E_trajectory_test run(s) per column)")
    display(mainablation_results(DB, p))

Main ablation results for problem=vanderpol (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,4.608e+00,3.292e+03,5.782e+00,5.052e+00,nan,nan
spec,5.901e-01,3.409e-01,3.842e-02,5.843e-01,nan,nan
VF,3.522e+01,4.110e+00,2.763e-01,1.124e+02,nan,nan
seed,"[21, 27]","[24, 28]","[27, 29]",[20],[],[]
n_params,"[317, 317]","[282, 282]","[268, 268]",[317],[],[]


Main ablation results for problem=pollu (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation
trajectory,1.923e+02,4.110e+36,6.942e-03
spec,5.367e+265,inf,6.920e+301
VF,2.876e+03,6.411e+35,8.929e+07
seed,"[25, 26]","[28, 22]","[22, 21]"
n_params,"[4940, 4940]","[4940, 4940]","[4744, 4744]"


Main ablation results for problem=rober (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


...

,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,nan,1.501e+02,1.014e+00,9.994e-01,nan,1.680e+00
spec,nan,1.025e+01,1.226e+00,1.892e+01,nan,1.471e+00
VF,nan,8.051e-01,8.017e-04,3.244e+00,nan,2.810e+02
seed,[],"[28, 20]","[27, 29]",[25],[],"[24, 20]"
n_params,[],"[516, 516]","[486, 486]",[516],[],"[486, 486]"


Main ablation results for problem=hires (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,6.957e+05,1.382e+09,2.177e+05,1.352e+05,3.021e+04,1.344e+04
spec,7.656e+05,9.050e+26,1.058e+27,9.975e-01,3.314e+31,3.419e+27
VF,1.263e+05,1.775e+07,4.054e+06,8.522e+04,2.667e+11,2.302e+07
seed,"[26, 27]","[27, 29]","[21, 26]","[21, 27]","[26, 28]",[26]
n_params,"[855, 855]","[827, 827]","[816, 816]","[855, 855]","[827, 827]",[816]


Main ablation results for problem=davis-skodje (dropped 2 best and 2 worst E_trajectory_test run(s) per column)


,GELU-scaled_collocation,mlp_collocation,stiff_collocation,GELU-scaled_shooting,mlp_shooting,stiff_shooting
trajectory,1.241e+05,5.848e+03,2.852e+12,5.657e+08,3.068e+08,1.240e+08
spec,9.745e-01,8.272e+00,5.014e-01,9.609e-01,1.619e+00,6.743e-01
VF,1.523e+02,3.025e+09,2.131e+06,3.121e+07,2.366e+13,3.285e+09
seed,"[28, 24]","[27, 26]","[22, 29]","[29, 27]","[29, 20]","[20, 28]"
n_params,"[317, 317]","[282, 282]","[268, 268]","[317, 317]","[282, 282]","[268, 268]"


In [92]:
def render_mainablation_row_group(table, problem_label="", model_order=("mlp", "GELU-scaled", "stiff"), training_order=("shooting", "collocation")):
    """Render the \\multirow Traj./Spec./VF body rows for one problem's
    mainablation_results table.

    The best (lowest) value in each row is bolded with \\textbf{}; missing or
    NaN values are rendered as "--".
    """
    columns = [f"{model}_{training}" for training in training_order for model in model_order]
    row_map = [("trajectory", "Traj."), ("spec", "Spec."), ("VF", "VF")]

    lines = [rf"\multirow{{{len(row_map)}}}{{*}}{{{problem_label}}}"]
    for row_key, row_label in row_map:
        cells, numeric = [], []
        for col in columns:
            raw = table.loc[row_key, col] if (row_key in table.index and col in table.columns) else None
            try:
                value = float(raw)
            except (TypeError, ValueError):
                value = math.nan
            if math.isnan(value):
                cells.append("--")
                numeric.append(math.inf)
            else:
                cells.append(str(raw))
                numeric.append(value)

        best_idx = numeric.index(min(numeric)) if numeric else None
        rendered = []
        for j, cell in enumerate(cells):
            if best_idx is not None and j == best_idx and cell != "--":
                cell = rf"\textbf{{{cell}}}"
            rendered.append(cell)

        lines.append(f"& {row_label} & " + " & ".join(rendered) + r" \\")

    return "\n".join(lines)


def render_mainablation_latex(table, problem_label="", label="tab:main-results",
                               model_order=("mlp", "GELU-scaled", "stiff"),
                               training_order=("shooting", "collocation")):
    """Take the output of mainablation_results for one problem and render the
    full LaTeX table (header + Traj./Spec./VF rows) with the numbers filled in.
    """
    header = [
        rf"\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll|ccc|ccc}",
        r"\toprule",
        r"& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\",
        r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
        r"\textbf{Problem} & \textbf{Metric}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet} \\",
        r"\midrule",
    ]
    body = render_mainablation_row_group(table, problem_label, model_order, training_order)
    footer = [r"\bottomrule", r"\end{tabular}", r"}"]
    return "\n".join(header + [body] + footer)


print(render_mainablation_latex(mainablation_results(DB, "vanderpol"), problem_label="Van der Pol"))

\label{tab:main-results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|ccc|ccc}
\toprule
& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
\textbf{Problem} & \textbf{Metric}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet} \\
\midrule
\multirow{3}{*}{Van der Pol}
& Traj. & -- & 5.459e+00 & 4.161e+01 & 3.352e+03 & \textbf{4.754e+00} & 8.225e+00 \\
& Spec. & -- & 5.766e+02 & 3.264e-01 & 2.880e-01 & 5.243e-01 & \textbf{5.427e-02} \\
& VF & -- & 2.099e+05 & 2.737e+02 & 4.148e+00 & 2.532e+01 & \textbf{6.065e-01} \\
\bottomrule
\end{tabular}
}


In [93]:
PROBLEM_DISPLAY = {
    "vanderpol": "Van der Pol",
    "pollu": "POLLU",
    "rober": "ROBER",
    "orego": "OREGO",
    "hires": "HIRES",
    "davis-skodje": "Davis--Skodje",
}

def render_mainablation_latex_all(db, problems=PROBLEMS, k=2,
                                   model_order=("mlp", "GELU-scaled", "stiff"),
                                   training_order=("shooting", "collocation"),
                                   problem_display=PROBLEM_DISPLAY,
                                   label="tab:main-results"):
    """Loop mainablation_results over `problems` and stack their row-groups
    into the full LaTeX table, separated by \\midrule.
    """
    header = [
        rf"\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll|ccc|ccc}",
        r"\toprule",
        r"& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\",
        r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}",
        r"\textbf{Problem} & \textbf{Metric}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet}",
        r"& \textbf{MLP}",
        r"& \textbf{GELU}",
        r"& \textbf{StiffNet} \\",
        r"\midrule",
    ]

    body = []
    for problem in problems:
        table = mainablation_results(db, problem, k=k, model_order=list(model_order), training_order=list(training_order))
        label_text = problem_display.get(problem, problem)
        body.append(render_mainablation_row_group(table, label_text, model_order, training_order))
        body.append(r"\midrule")
    if body and body[-1] == r"\midrule":
        body.pop()  # no trailing midrule after the last problem

    footer = [r"\bottomrule", r"\end{tabular}", r"}"]
    return "\n".join(header + body + footer)


print(render_mainablation_latex_all(DB))

\label{tab:main-results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|ccc|ccc}
\toprule
& & \multicolumn{3}{c|}{\textbf{Single Shooting}} & \multicolumn{3}{c}{\textbf{Collocation}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
\textbf{Problem} & \textbf{Metric}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet}
& \textbf{MLP}
& \textbf{GELU}
& \textbf{StiffNet} \\
\midrule
\multirow{3}{*}{Van der Pol}
& Traj. & 2.756e+03 & \textbf{5.009e+00} & 6.221e+01 & 3.375e+03 & 8.919e+00 & 3.352e+01 \\
& Spec. & 3.907e-01 & 3.462e+02 & 4.636e-01 & 3.430e-01 & 5.728e-01 & \textbf{5.678e-02} \\
& VF & 8.498e+01 & 1.284e+05 & 4.483e+02 & 3.521e+00 & 2.219e+01 & \textbf{5.635e-01} \\
\midrule
\multirow{3}{*}{POLLU}
& Traj. & -- & -- & -- & 3.596e+36 & 2.531e+02 & \textbf{2.472e-02} \\
& Spec. & -- & -- & -- & inf & \textbf{1.379e+268} & 3.319e+302 \\
& VF & -- & -- & -- & 2.137e+35 & \textbf{3.574e+03} & 9.545e+07 \\
\midrule
\multirow{3}{*}{ROBER}
& Traj. & 4.281e+04 & 2.326e+21 & \textbf{1.771e+01} & 

In [94]:
DB["seed"].unique()
# DB["pretraining"].unique()
# DB["training"].unique()

array([25, 24, 23, 26, 27, 22, 21, 28, 29, 20])